In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf

In [2]:
socialMedia = pd.read_csv("Social_media_impact_on_life.csv")

In [3]:
socialMediaLabels = socialMedia["Overall_Impact"]
socialMediaVariables = socialMedia.drop(columns=["Student_ID","Overall_Impact"])

In [4]:
#one-hot encoding
print(socialMediaVariables["Gender"].unique())
print(socialMediaVariables["Primary_Platform"].unique())
print(socialMediaVariables["Device_Type"].unique())
#dummy encoding
print(socialMediaVariables["Academic_Level"].unique())
print(socialMediaVariables["Social_Comparison_Frequency"].unique())

['Female' 'Male' 'Non-Binary' 'Prefer not to say']
['Snapchat' 'Instagram' 'TikTok' 'YouTube' 'LinkedIn' 'Reddit'
 'X (Twitter)']
['Smartphone' 'Tablet' 'Laptop/PC']
['Undergraduate' 'High School' 'Postgraduate']
['Always' 'Sometimes' 'Frequently' 'Never' 'Rarely']


In [5]:
socialMediaVariables = pd.get_dummies(socialMediaVariables, columns=['Gender', 'Primary_Platform', 'Device_Type', 'Academic_Level'])

In [6]:
def consistencyTranslate(x):
    if x == 'Never':
        return 0
    elif x == 'Rarely':
        return 1
    elif x == 'Sometimes':
        return 2
    elif x == 'Frequently':
        return 3
    else:
        return 4

In [7]:
socialMediaVariables['Social_Comparison_Frequency'] = socialMediaVariables['Social_Comparison_Frequency'].map(consistencyTranslate)

In [8]:
def labelTranslate(x):
    if x == 'Negative':
        return -1
    elif x == 'Neutral':
        return 0
    else:
        return 1

In [9]:
labelMapping = {"Negative": 0, "Neutral": 1, "Beneficial": 2}
socialMediaLabels = socialMediaLabels.map(labelMapping)

In [10]:
socialMediaVariables.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Age                           4500 non-null   int64  
 1   Daily_Usage_Hours             4500 non-null   float64
 2   Weekend_Extra_Hours           4500 non-null   float64
 3   Sleep_Duration_Hours          4500 non-null   float64
 4   Sleep_Quality_Score           4500 non-null   int64  
 5   Late_Night_Usage              4500 non-null   bool   
 6   Social_Comparison_Frequency   4500 non-null   int64  
 7   Perceived_Stress_Score        4454 non-null   float64
 8   Mental_Health_Index           4500 non-null   int64  
 9   Academic_Performance_GPA      4415 non-null   float64
 10  Gender_Female                 4500 non-null   bool   
 11  Gender_Male                   4500 non-null   bool   
 12  Gender_Non-Binary             4500 non-null   bool   
 13  Gen

In [11]:
X_train, X_test, y_train, y_test = train_test_split(socialMediaVariables, socialMediaLabels, test_size=0.2, random_state=42)

In [39]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(27,)),
    tf.keras.layers.Dense(2700, activation='relu'),
    tf.keras.layers.Dropout(.5),
    tf.keras.layers.Dense(2700, activation='relu'),
    tf.keras.layers.Dense(3)
])
model.compile(optimizer='adamw',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

C:\Users\harri\anaconda3\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [40]:
model.fit(X_train, y_train, epochs=3)

Epoch 1/3
106/106 ━━━━━━━━━━━━━━━━━━━━ 17s 122ms/step - accuracy: 0.8077 - loss: 0.7300
Epoch 2/3
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 125ms/step - accuracy: 0.8130 - loss: 0.5759
Epoch 3/3
106/106 ━━━━━━━━━━━━━━━━━━━━ 13s 126ms/step - accuracy: 0.8130 - loss: 0.5762


In [41]:
test_loss, test_acc = model.evaluate(X_test,  y_test, verbose=2)

print('\nTest accuracy:', test_acc)

36/36 - 1s - 23ms/step - accuracy: 0.8329 - loss: 0.5399

Test accuracy: 0.8328889012336731
